# CODEx Single‑Sample Analysis Demo
This notebook demonstrates how to:
1. **Load** spatial CODEx datasets (UPMC, Charville, DFCI)
2. **Inspect** available cell‑type annotations
3. **Define** tumor centre mappings for feature extraction
4. **Run** the TIC single‑sample analysis pipeline
5. **Visualise** cross‑region causal effects with automatic pseudo‑time direction correction

> *Update the `EXP_ROOT` path to your own workspace before executing.*

In [18]:
import warnings, os, pickle
warnings.filterwarnings('ignore')

from tic.data import list_regions, load_region

from pathlib import Path
from typing import List
from tqdm import tqdm
from tic.plotting import plot_monotonicity_metrics_bar, plot_biomarker_trends
from tic.wrappers.feature import FeatureWrapper
from tic.wrappers.pseudotime import PseudotimeWrapper
from tic.wrappers.causal import CausalWrapper
from utils.dataset.codex_upmc import UPMC_EMT_GENES



In [19]:
# root directory to save experiment outputs
EXP_ROOT = "../tutorial/example_out/multi_regions"  # <-- EDIT ME

# enumerate available regions
upmc_regions      = list_regions(dataset='upmc')
charville_regions = list_regions(dataset='charville')
dfci_regions      = list_regions(dataset='dfci')

print(f"UPMC      regions: {len(upmc_regions)}")
print(f"Charville regions: {len(charville_regions)}")
print(f"DFCI      regions: {len(dfci_regions)}")


2025-05-30 10:18:10,903 INFO Extracting upmc_raw_data.zip → /Users/zhangjiahao/.cache/tic/codex_upmc
2025-05-30 10:18:10,903 | INFO     | tic.data.io | Extracting upmc_raw_data.zip → /Users/zhangjiahao/.cache/tic/codex_upmc


Skipping download; file exists: /Users/zhangjiahao/.cache/tic/codex_upmc/upmc_raw_data.zip


2025-05-30 10:18:15,717 INFO Extraction complete: /Users/zhangjiahao/.cache/tic/codex_upmc
2025-05-30 10:18:15,717 | INFO     | tic.data.io | Extraction complete: /Users/zhangjiahao/.cache/tic/codex_upmc
Flattening nested directory: /Users/zhangjiahao/.cache/tic/codex_upmc/raw_data
2025-05-30 10:18:15,720 | INFO     | tic.data.codex.download | Flattening nested directory: /Users/zhangjiahao/.cache/tic/codex_upmc/raw_data
Removed nested directory: /Users/zhangjiahao/.cache/tic/codex_upmc/raw_data
2025-05-30 10:18:16,058 | INFO     | tic.data.codex.download | Removed nested directory: /Users/zhangjiahao/.cache/tic/codex_upmc/raw_data
2025-05-30 10:18:16,062 INFO Extracting charville_raw_data.zip → /Users/zhangjiahao/.cache/tic/codex_charville
2025-05-30 10:18:16,062 | INFO     | tic.data.io | Extracting charville_raw_data.zip → /Users/zhangjiahao/.cache/tic/codex_charville


Skipping download; file exists: /Users/zhangjiahao/.cache/tic/codex_charville/charville_raw_data.zip


2025-05-30 10:18:17,884 INFO Extraction complete: /Users/zhangjiahao/.cache/tic/codex_charville
2025-05-30 10:18:17,884 | INFO     | tic.data.io | Extraction complete: /Users/zhangjiahao/.cache/tic/codex_charville
Flattening nested directory: /Users/zhangjiahao/.cache/tic/codex_charville/raw_data
2025-05-30 10:18:17,885 | INFO     | tic.data.codex.download | Flattening nested directory: /Users/zhangjiahao/.cache/tic/codex_charville/raw_data
Removed nested directory: /Users/zhangjiahao/.cache/tic/codex_charville/raw_data
2025-05-30 10:18:18,053 | INFO     | tic.data.codex.download | Removed nested directory: /Users/zhangjiahao/.cache/tic/codex_charville/raw_data
2025-05-30 10:18:18,056 INFO Extracting dfci_raw_data.zip → /Users/zhangjiahao/.cache/tic/codex_dfci
2025-05-30 10:18:18,056 | INFO     | tic.data.io | Extracting dfci_raw_data.zip → /Users/zhangjiahao/.cache/tic/codex_dfci


Skipping download; file exists: /Users/zhangjiahao/.cache/tic/codex_dfci/dfci_raw_data.zip


2025-05-30 10:18:18,521 INFO Extraction complete: /Users/zhangjiahao/.cache/tic/codex_dfci
2025-05-30 10:18:18,521 | INFO     | tic.data.io | Extraction complete: /Users/zhangjiahao/.cache/tic/codex_dfci
Flattening nested directory: /Users/zhangjiahao/.cache/tic/codex_dfci/raw_data
2025-05-30 10:18:18,522 | INFO     | tic.data.codex.download | Flattening nested directory: /Users/zhangjiahao/.cache/tic/codex_dfci/raw_data
Removed nested directory: /Users/zhangjiahao/.cache/tic/codex_dfci/raw_data
2025-05-30 10:18:18,555 | INFO     | tic.data.codex.download | Removed nested directory: /Users/zhangjiahao/.cache/tic/codex_dfci/raw_data


UPMC      regions: 308
Charville regions: 292
DFCI      regions: 58


In [20]:
print(upmc_regions[0])

UPMC_c001_v001_r001_reg001


In [21]:
# Load one example region per dataset to inspect cell types
upmc_example      = load_region('upmc', region_id=upmc_regions[0])
charville_example = load_region('charville', region_id=charville_regions[0])
dfci_example      = load_region('dfci', region_id=dfci_regions[0])

def tumor_types(adata):
    cts = adata.obs['cell_type'].unique()
    return [ct for ct in cts if 'tumor' in ct.lower()]

print("Tumor cell types:")
print("  UPMC      :", tumor_types(upmc_example))
print("  Charville :", tumor_types(charville_example))
print("  DFCI      :", tumor_types(dfci_example))

# mapping used by FeatureWrapper
Tumor_Cell_Mapping = {
    "upmc": tumor_types(upmc_example),
    "charville": tumor_types(charville_example),
    "dfci": tumor_types(dfci_example),
}


2025-05-30 10:18:18,588 INFO Extracting upmc_raw_data.zip → /Users/zhangjiahao/.cache/tic/codex_upmc
2025-05-30 10:18:18,588 | INFO     | tic.data.io | Extracting upmc_raw_data.zip → /Users/zhangjiahao/.cache/tic/codex_upmc


Skipping download; file exists: /Users/zhangjiahao/.cache/tic/codex_upmc/upmc_raw_data.zip


2025-05-30 10:18:22,756 INFO Extraction complete: /Users/zhangjiahao/.cache/tic/codex_upmc
2025-05-30 10:18:22,756 | INFO     | tic.data.io | Extraction complete: /Users/zhangjiahao/.cache/tic/codex_upmc
Flattening nested directory: /Users/zhangjiahao/.cache/tic/codex_upmc/raw_data
2025-05-30 10:18:22,761 | INFO     | tic.data.codex.download | Flattening nested directory: /Users/zhangjiahao/.cache/tic/codex_upmc/raw_data
Removed nested directory: /Users/zhangjiahao/.cache/tic/codex_upmc/raw_data
2025-05-30 10:18:22,979 | INFO     | tic.data.codex.download | Removed nested directory: /Users/zhangjiahao/.cache/tic/codex_upmc/raw_data
2025-05-30 10:18:23,013 INFO Extracting charville_raw_data.zip → /Users/zhangjiahao/.cache/tic/codex_charville
2025-05-30 10:18:23,013 | INFO     | tic.data.io | Extracting charville_raw_data.zip → /Users/zhangjiahao/.cache/tic/codex_charville


Skipping download; file exists: /Users/zhangjiahao/.cache/tic/codex_charville/charville_raw_data.zip


2025-05-30 10:18:24,861 INFO Extraction complete: /Users/zhangjiahao/.cache/tic/codex_charville
2025-05-30 10:18:24,861 | INFO     | tic.data.io | Extraction complete: /Users/zhangjiahao/.cache/tic/codex_charville
Flattening nested directory: /Users/zhangjiahao/.cache/tic/codex_charville/raw_data
2025-05-30 10:18:24,862 | INFO     | tic.data.codex.download | Flattening nested directory: /Users/zhangjiahao/.cache/tic/codex_charville/raw_data
Removed nested directory: /Users/zhangjiahao/.cache/tic/codex_charville/raw_data
2025-05-30 10:18:24,998 | INFO     | tic.data.codex.download | Removed nested directory: /Users/zhangjiahao/.cache/tic/codex_charville/raw_data
2025-05-30 10:18:25,017 INFO Extracting dfci_raw_data.zip → /Users/zhangjiahao/.cache/tic/codex_dfci
2025-05-30 10:18:25,017 | INFO     | tic.data.io | Extracting dfci_raw_data.zip → /Users/zhangjiahao/.cache/tic/codex_dfci


Skipping download; file exists: /Users/zhangjiahao/.cache/tic/codex_dfci/dfci_raw_data.zip


2025-05-30 10:18:25,393 INFO Extraction complete: /Users/zhangjiahao/.cache/tic/codex_dfci
2025-05-30 10:18:25,393 | INFO     | tic.data.io | Extraction complete: /Users/zhangjiahao/.cache/tic/codex_dfci
Flattening nested directory: /Users/zhangjiahao/.cache/tic/codex_dfci/raw_data
2025-05-30 10:18:25,394 | INFO     | tic.data.codex.download | Flattening nested directory: /Users/zhangjiahao/.cache/tic/codex_dfci/raw_data
Removed nested directory: /Users/zhangjiahao/.cache/tic/codex_dfci/raw_data
2025-05-30 10:18:25,420 | INFO     | tic.data.codex.download | Removed nested directory: /Users/zhangjiahao/.cache/tic/codex_dfci/raw_data


Tumor cell types:
  UPMC      : ['Tumor (Ki67+)', 'Tumor (CD20+)', 'Tumor (CD21+)', 'Tumor (Podo+)', 'Tumor', 'Tumor (CD15+)']
  Charville : ['Tumor 2 (Ki67 Proliferating)', 'Tumor 3', 'Tumor 7', 'Tumor 4', 'Tumor 6 / DC', 'Tumor 1', 'Tumor 5']
  DFCI      : ['Tumor (PanCK hi)', 'Tumor (PanCK low)', 'Tumor (Ki67+)', 'Tumor (CD15+)']


In [22]:
print('gene names per dataset:')
print('UPMC     :', upmc_example.var_names.tolist())
print('Charville:', charville_example.var_names.tolist())
print('DFCI     :', dfci_example.var_names.tolist())


gene names per dataset:
UPMC     : ['CD11b', 'CD14', 'CD15', 'CD163', 'CD20', 'CD21', 'CD31', 'CD34', 'CD3e', 'CD4', 'CD45', 'CD45RA', 'CD45RO', 'CD68', 'CD8', 'CollagenIV', 'HLA-DR', 'Ki67', 'PanCK', 'Podoplanin', 'Vimentin', 'aSMA']
Charville: ['CD107a', 'CD117', 'CD11b', 'CD11c', 'CD134', 'CD14', 'CD15', 'CD163', 'CD20', 'CD21', 'CD31', 'CD38', 'CD3e', 'CD4', 'CD45', 'CD45RA', 'CD45RO', 'CD49f', 'CD68', 'CD8', 'CollagenIV', 'DAPI', 'FoxP3', 'Gal3', 'GranzymeB', 'HLA-DR', 'ICOS', 'Ki67', 'PD1', 'PDL1', 'PGP9.5', 'PanCK', 'Podoplanin', 'RORgammaT', 'S100A4', 'Siglec8', 'TFAM', 'TIM3', 'Vimentin', 'aSMA']
DFCI     : ['CD103', 'CD117', 'CD11b', 'CD11c', 'CD14', 'CD15', 'CD163', 'CD183', 'CD197', 'CD20', 'CD25', 'CD31', 'CD38', 'CD39', 'CD3e', 'CD4', 'CD45', 'CD45RA', 'CD45RO', 'CD56', 'CD68', 'CD69', 'CD8', 'DAPI', 'FoxP3', 'GranzymeB', 'GranzymeK', 'HLA-ABC', 'HLA-DR', 'ICOS', 'Ki67', 'LAG3', 'PD1', 'PDL1', 'PanCK', 'Perforin', 'Siglec8', 'TCF1', 'TCRgammadelta', 'TIM3', 'Vimentin']


## Single‑Sample Analysis Pipeline
We will compute:
1. **Cell‑type composition features** (`recipe='tme_default'`).
2. **Pseudotime** (`TIC.PseudotimeWrapper`).
3. **Biomarker trends** over pseudotime for EMT markers.
4. **Granger causality** of cell‑type predictors to target biomarkers.

Outputs are saved under `EXP_ROOT/<dataset>/<region_id>/`.

In [23]:
def single_sample_analysis(dataset: str, region_ids: List[str], out_dir: str, method: str = 'composition') -> None:
    """
    Run TIC single-sample workflow for multiple regions.
    For each region, extract features and perform single-sample analysis:
        1. Feature extraction
        2. Pseudotime inference
        3. Biomarker trend plotting
        4. Causal inference analysis
        5. Result saving

    Output structure:
    out_dir/
        region_id/
            biomarker_trends.png
            metrics.csv
            causal/
                biomarker/
                    bar.png
                    volcano.png
                    heatmap.png
                    results.pkl
    """
    os.makedirs(out_dir, exist_ok=True)

    for rid in tqdm(region_ids, desc=f'[{dataset}]'):
        adata  = load_region(dataset=dataset, region_id=rid)
        genes  = set(adata.var_names)
        rdir   = os.path.join(out_dir, rid)
        os.makedirs(rdir, exist_ok=True)

        # Feature extraction
        fw = FeatureWrapper(
            recipe='tme_default',
            centre_types=Tumor_Cell_Mapping[dataset],
            graph_params={'method': 'voronoi'},
            subgraph_params={'strategy': 'radius', 'radius': 75},
        )
        fea_adata = fw.fit(adata)

        # Pseudotime inference
        ptw      = PseudotimeWrapper(rep_key=method, output_dir=rdir)
        pt_adata = ptw.fit(fea_adata)
        ptw.metrics.to_csv(os.path.join(rdir, 'metrics.csv'), index=False)
        
        plot_monotonicity_metrics_bar(metrics=ptw.metrics)

        # Biomarker trends
        plot_biomarker_trends(
            adata=pt_adata,
            selected_biomarkers=UPMC_EMT_GENES,
            x_transform='bin+normalize',
            y_transform='normalize',
            bins=100,
            save_path=os.path.join(rdir, 'biomarker_trends.png'),
        )

        # Causal inference
        causal_root = Path(rdir) / "causal"
        causal_root.mkdir(exist_ok=True)
        for biomarker in set(UPMC_EMT_GENES) & set(genes):
            dest = causal_root / biomarker
            dest.mkdir(exist_ok=True)

            cw = CausalWrapper(
                outcome=biomarker,
                include_extractors=('composition',),
                method='granger_causality',
                bins=100,
                method_kwargs={'maxlag': 3},
            )
            c_adata = cw.fit(pt_adata)
            cw.plot(c_adata, kind='bar',      save_path=dest/'bar.png')
            cw.plot(c_adata, kind='volcano',  save_path=dest/'volcano.png')
            cw.plot(c_adata, kind='heatmap',  save_path=dest/'heatmap.png')

            with open(dest/'results.pkl', 'wb') as fh:
                pickle.dump(cw.results, fh)


In [ ]:
single_sample_analysis(dataset='upmc', region_ids=upmc_regions[0], out_dir=EXP_ROOT, method='composition')

## Cross‑Region Causal Heatmap
To compare causal effects across regions while accounting for pseudo‑time direction (EMT vs MET), we use `plot_cross_region_causal_heatmap`. Specify `expected_sign` for the target biomarker:

* **E‑markers** → expected positive correlation
* **M‑markers** → expected negative correlation

In [ ]:
from tic.plotting import plot_cross_region_causal_heatmap

In [ ]:
# >>> Example (after analysis is done)
plot_cross_region_causal_heatmap(
    out_dir=os.path.expanduser("~")+'/Expriments/Codex/charville',
    region_ids=charville_regions,
    y_var='PanCK',
    expected_sign='negative',   # PanCK expected to decrease along EMT
    top_n=100,
    top_strategy='median',
    metric='rms',
    log2_transform=False,
    row_cluster=True,
    col_cluster=True,
    title="Causal effect on 'PanCK' (metric: rms, p ≤ 0.05) Data from Charville",
    # save_path=str(TEST_DIR/'PanCK_heatmap.png'),
)


In [ ]:
# >>> Example (after analysis is done)
plot_cross_region_causal_heatmap(
    out_dir=os.path.expanduser("~")+'/Expriments/Codex/dfci',
    region_ids=dfci_regions,
    y_var='PanCK',
    expected_sign='negative',   # PanCK expected to decrease along EMT
    metric='rms',
    log2_transform=False,
    title="Causal effect on 'PanCK' (metric: rms, p ≤ 0.05) Data from DFCI",
    # save_path=str(TEST_DIR/'PanCK_heatmap.png'),
)


In [ ]:
# >>> Example (after analysis is done)
plot_cross_region_causal_heatmap(
    out_dir=os.path.expanduser("~")+'/Expriments/Codex/upmc',
    region_ids=upmc_regions,
    y_var='PanCK',
    expected_sign='negative',   # PanCK expected to decrease along EMT
    metric='rms',
    log2_transform=False,
    title="Causal effect on 'PanCK' (metric: rms, p ≤ 0.05) Data from UPMC",
    # save_path=str(TEST_DIR/'PanCK_heatmap.png'),
)
